# Einsum

Based on 
- https://rockt.ai/2018/04/30/einsum (Einsum is All you Need - Einstein Summation in Deep Learning)
- https://cxtraa.github.io/machine-learning/einops-arena.html (Einops is all you need)


`einsum` allows us to compute things like inner/outer products and multiply tensors together. Here are the basic rules:

- If a dimension is included in the input tensors but not the output, then this implies summation over that dimension.
- If a dimension appears in the input and output tensors, then we multiply this dimension as normal.

Libraries like `numpy` and `pytorch` provide some implementation of `einsum()`. `einops` also has an `einsum` function that
works across common frameworks.


In [40]:
import torch
a = torch.arange(6).reshape(2, 3)
a

tensor([[0, 1, 2],
        [3, 4, 5]])

In [41]:
# matrix transponse
torch.einsum("ij -> ji", [a]) # all input indices are included in the output, no summation/multiplication performed, but transposed due to reversed order

tensor([[0, 3],
        [1, 4],
        [2, 5]])

In [42]:
# same operation using einops
import einops
einops.rearrange(a, "i j -> j i")

tensor([[0, 3],
        [1, 4],
        [2, 5]])

In [43]:
# you can also use einops.einsum here
einops.einsum(a, "i j -> j i")

tensor([[0, 3],
        [1, 4],
        [2, 5]])

In [44]:
# Sum
torch.einsum("ij -> ", a) # All input indices are omitted from output, all elements are summed. No mulitiplication performed since it's a single input?

tensor(15)

In [45]:
# Same using einops.einsum
einops.einsum(a, "i j -> ") 

tensor(15)

In [46]:
# More explicitly, using einops.reduce
einops.reduce(a, "i j -> ()", "sum")

tensor([15])

In [47]:
# Column sum

print(torch.einsum("ij -> j", [a])) # row index omitted, column index preserved -> sum over rows for each column
print(einops.einsum(a, "i j -> j"))
print(einops.reduce(a, "i j -> j", "sum"))

tensor([3, 5, 7])
tensor([3, 5, 7])
tensor([3, 5, 7])


In [48]:
# Row sum

print(torch.einsum("ij -> i", [a])) # row index preserved, column index omitted -> sum over columns for each row
print(einops.einsum(a, "i j -> i"))
print(einops.reduce(a, "i j -> i", "sum"))

tensor([ 3, 12])
tensor([ 3, 12])
tensor([ 3, 12])


In [49]:
# Matrix vector multiplication

b = torch.arange(3)
print("a =", a)
print("b =", b)

print("Intermediate products before per-row sums")
print(torch.einsum("ij,j->i j", [a, b]))
print(torch.einsum("ij,j->i", [a, b])) # a[i,j] is multiplied by b[j] for each i, j, then for each row i, the intermediate products are summed over j
print(einops.einsum(a, b, "i j, j -> i"))

a = tensor([[0, 1, 2],
        [3, 4, 5]])
b = tensor([0, 1, 2])
Intermediate products before per-row sums
tensor([[ 0,  1,  4],
        [ 0,  4, 10]])
tensor([ 5, 14])
tensor([ 5, 14])


In [50]:
print(a)
print(b)
# Elementwise multiplication then sum over columns
print(a * b)
print(einops.reduce(a * b, "i j -> i", "sum"))


tensor([[0, 1, 2],
        [3, 4, 5]])
tensor([0, 1, 2])
tensor([[ 0,  1,  4],
        [ 0,  4, 10]])
tensor([ 5, 14])


In [51]:
# Matrix-matrix multiplication
x = torch.arange(6).reshape(2, 3)
y = torch.arange(15).reshape(3, 5)
torch.einsum("ik,kj->ij", [x, y]) # Multiply x[i,k] by y[k,j] for each i, j, k, then sum over k for each i, j

tensor([[ 25,  28,  31,  34,  37],
        [ 70,  82,  94, 106, 118]])

In [52]:
einops.einsum(x, y, "i k, k j -> i j")

tensor([[ 25,  28,  31,  34,  37],
        [ 70,  82,  94, 106, 118]])

In [53]:
# To achieve the same with einops.reduce, it gets a little tricky
# We need to first rearrange the tensors to have the same shape, then do an elementwise multiplication, then reduce
einops.reduce(
    einops.rearrange(x, "i k -> i k ()") * einops.rearrange(y, "k j -> () k j"),
    "i k j -> i j",
    "sum"
)

tensor([[ 25,  28,  31,  34,  37],
        [ 70,  82,  94, 106, 118]])

In [54]:
# Dot product
m = torch.arange(3)
n = torch.arange(3, 6)
print("m =", m)
print("n =", n)

print(torch.einsum("i,i->", [m, n]))
print(einops.einsum(m, n, "i, i -> "))
print(einops.reduce(m * n, "i -> ", "sum"))


m = tensor([0, 1, 2])
n = tensor([3, 4, 5])
tensor(14)
tensor(14)
tensor(14)


In [55]:
# Outer product

# The outer product produces a matrix of all possible products of scalar values in vec1 and vec2.
# It involves no summation, so no variables are omitted in the output.
# let N = len(vec1), M = len(vec2), then len of outer product = N x M
# Outer[i,j] = vec1[i] * vec2[j]

o = torch.arange(3)
p = torch.arange(3, 7) # vector of length 4
print("o =", o)
print("p =", p)

print(einops.einsum(o, p, "i,j -> i j")) # We multiply each element of o to each element of p, resulting in a i*j matrix. No summation since all indices are preserved


o = tensor([0, 1, 2])
p = tensor([3, 4, 5, 6])
tensor([[ 0,  0,  0,  0],
        [ 3,  4,  5,  6],
        [ 6,  8, 10, 12]])


In [69]:
# Hadamard product

# The Hadamard product takes two matrices of the same dimensions and returns a matrix of the multiplied corresponding elements.
# It's basically element-wise multiplication of two matrics

r = torch.arange(6).reshape(2, 3)
s = torch.arange(6,12).reshape(2, 3)
print("r:")
print(r)
print("s:")
print(s)

print(einops.einsum(r, s, "i j, i j -> i j"))
print(r * s)

r:
tensor([[0, 1, 2],
        [3, 4, 5]])
s:
tensor([[ 6,  7,  8],
        [ 9, 10, 11]])
tensor([[ 0,  7, 16],
        [27, 40, 55]])
tensor([[ 0,  7, 16],
        [27, 40, 55]])


In [ ]:
# Batch matrix multiplication
torch.manual_seed(42)
g = torch.randn(3, 2, 5)
b = torch.randn(3, 5, 3)

einops.einsum(g, b, "b i k,b k j -> b i j")

tensor([[[-0.8520,  3.9174, -2.3697],
         [ 3.0188,  0.6928,  3.1418]],

        [[-4.0935, -1.6975, -4.9236],
         [-1.8561, -1.5185, -1.1148]],

        [[-0.8815, -0.5478, -2.2570],
         [ 0.8440,  2.1193, -1.8171]]])

In [ ]:
# Tensor contraction

# Batch matrix multiplication is a special case of a tensor contraction (see: https://en.wikipedia.org/wiki/Tensor_contraction).
# Let's say we have two tensors, an order-n tensor A with dimensions I1, I2, ... In tensor and an order-m tensor B with dimensions J1, J2, ... Jm.
# As an example, take n=4, m=5
# and assume that I2=J3 and I3=J5
# We can multiply the two tensors in these two dimensions (2 and 3 for A and 3 and 5 for B) resulting in a new tensor C∈ with dimensions I1, I4, J1, J2, J4
# as follows:
A = torch.randn(2, 3, 5, 7)
B = torch.randn(11, 13, 3, 17, 5)

einops.einsum(A, B, "p q r s, t u q v r -> p s t u v").shape

torch.Size([2, 7, 11, 13, 17])

In [67]:
# Bilinear transformation

# einsum can operate on more than tensors. One example where this is ued is bilinear transformation (see: http://pytorch.org/docs/master/nn.html#torch.nn.Bilinear)

a = torch.randn(2,3)
b = torch.randn(5,3,7)
c = torch.randn(2,7)
print(torch.einsum('ik,jkl,il->ij', [a, b, c]))
print(einops.einsum(a, b, c, 'i k, j k l,i l->i j'))

tensor([[ 1.3137, -2.2561,  4.0544, -2.0511, -0.7363],
        [ 2.9287,  0.6450,  6.1502,  1.0987, -5.6000]])
tensor([[ 1.3137, -2.2561,  4.0544, -2.0511, -0.7363],
        [ 2.9287,  0.6450,  6.1502,  1.0987, -5.6000]])
